In [1]:
# ================================
# 1. Imports
# ================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, round

# ================================
# 2. Create Spark Session (CLUSTER MODE)
# ================================
spark = (
    SparkSession.builder
    .appName("Orders Analysis")
    .master("spark://spark-master:7077")   # ✅ connect to cluster
    .config("spark.executor.instances", "2")
    .config("spark.executor.cores", "1")
    .getOrCreate()
)

# ================================
# 3. Read CSV (FINAL PATH ✅)
# ================================
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/data/data/orders_100mb.csv")   # ✅ SAME PATH FOR ALL CONTAINERS
)

# ================================
# 4. Validate Data
# ================================
df.printSchema()
df.show(5)

# ================================
# 5. Transformation (your logic)
# ================================
df_result = (
    df.filter(col("state") == "Uttar Pradesh")
    .withColumn("sales", col("price") * col("quantity"))
    .groupBy("product_id")
    .agg(round(sum("sales"), 2).alias("total_sales"))
    .orderBy(col("total_sales").desc())
    .limit(5)
)

# ================================
# 6. Show Result
# ================================
df_result.show()

# ================================
# 7. Verify Distributed Execution (IMPORTANT)
# ================================
print(spark.sparkContext.getExecutorMemoryStatus())

# ================================
# 8. Stop Spark (optional)
# ================================
# spark.stop()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- state: string (nullable = true)
 |-- quantity: integer (nullable = true)

+--------+-----------+----------+------+----------+------------+-------------+--------+
|order_id|customer_id|product_id| price|order_date|order_status|        state|quantity|
+--------+-----------+----------+------+----------+------------+-------------+--------+
|       1|     269253|        45|188.67|2002-08-26|      PLACED|Uttar Pradesh|       2|
|       2|     428361|        86|191.63|2023-10-03|      PLACED|    Rajasthan|       2|
|       3|     544257|        17|135.49|2019-12-16|      PLACED|        Assam|       3|
|       4|     450285|        60|135.16|2009-03-15|     SHIPPED|        Delhi|       2|
|       5|     916968|        32|171.31|2002-03-19|

AttributeError: 'SparkContext' object has no attribute 'getExecutorMemoryStatus'